# 2. Data Cleaning & Preparation

## 2.1 Setup & Data Loading

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# pd.set_option("display.max_rows", None)
# pd.set_option('display.max_columns', None)
# pd.reset_option('display.max_rows')

In [3]:
RAW_DIR = Path("../data/raw")

In [4]:
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
csv_files = list(RAW_DIR.glob("*.csv"))

In [6]:
print(f"Total files: {len(csv_files)}")
for file in csv_files:
    size = ((file.stat().st_size) / (1024 * 1024))
    print(f"{file.name} | {size:.2f}MB")

Total files: 9
olist_customers_dataset.csv | 8.62MB
olist_geolocation_dataset.csv | 58.44MB
olist_orders_dataset.csv | 16.84MB
olist_order_items_dataset.csv | 14.72MB
olist_order_payments_dataset.csv | 5.51MB
olist_order_reviews_dataset.csv | 13.78MB
olist_products_dataset.csv | 2.27MB
olist_sellers_dataset.csv | 0.17MB
product_category_name_translation.csv | 0.00MB


In [7]:
customers = pd.read_csv(RAW_DIR / "olist_customers_dataset.csv")
geolocation = pd.read_csv(RAW_DIR / "olist_geolocation_dataset.csv")
orders = pd.read_csv(RAW_DIR / "olist_orders_dataset.csv")
order_items = pd.read_csv(RAW_DIR / "olist_order_items_dataset.csv")
order_payments = pd.read_csv(RAW_DIR / "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(RAW_DIR / "olist_order_reviews_dataset.csv")
products = pd.read_csv(RAW_DIR / "olist_products_dataset.csv")
sellers = pd.read_csv(RAW_DIR / "olist_sellers_dataset.csv")
product_category_name = pd.read_csv(RAW_DIR / "product_category_name_translation.csv")

## 2.2 Fixing Data Types (Timestamp)

### 2.2.1 Orders Timestamp From `str` To `datetime64`

In [8]:
order_cols = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
             "order_delivered_customer_date", "order_estimated_delivery_date"]

for col in order_cols:
   orders[col] = pd.to_datetime(orders[col])

In [9]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


### 2.2.2 Order Items Timestamp From `str` To `datetime64`

In [10]:
order_items["shipping_limit_date"] = pd.to_datetime(order_items["shipping_limit_date"]) 

In [11]:
order_items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


### 2.2.3 Order Reviews Timestamp From `str` To `datetime64`

In [12]:
review_col = ["review_creation_date", "review_answer_timestamp"]

for col in review_col:
    order_reviews[col] = pd.to_datetime(order_reviews[col])

In [13]:
order_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 99224 entries, 0 to 99223
Data columns (total 7 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   review_id                99224 non-null  str           
 1   order_id                 99224 non-null  str           
 2   review_score             99224 non-null  int64         
 3   review_comment_title     11568 non-null  str           
 4   review_comment_message   40977 non-null  str           
 5   review_creation_date     99224 non-null  datetime64[us]
 6   review_answer_timestamp  99224 non-null  datetime64[us]
dtypes: datetime64[us](2), int64(1), str(4)
memory usage: 5.3 MB


## Cleaning Product Category & Translation

In [14]:
products.info()

<class 'pandas.DataFrame'>
RangeIndex: 32951 entries, 0 to 32950
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   product_id                  32951 non-null  str    
 1   product_category_name       32341 non-null  str    
 2   product_name_lenght         32341 non-null  float64
 3   product_description_lenght  32341 non-null  float64
 4   product_photos_qty          32341 non-null  float64
 5   product_weight_g            32949 non-null  float64
 6   product_length_cm           32949 non-null  float64
 7   product_height_cm           32949 non-null  float64
 8   product_width_cm            32949 non-null  float64
dtypes: float64(7), str(2)
memory usage: 2.3 MB


In [15]:
products[products.isna().any(axis=1)]

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
105,a41e356c76fab66334f36de622ecbd3a,NaN,NaN,NaN,NaN,650.0,17.0,14.0,12.0
128,d8dee61c2034d6d075997acef1870e9b,NaN,NaN,NaN,NaN,300.0,16.0,7.0,20.0
145,56139431d72cd51f19eb9f7dae4d1617,NaN,NaN,NaN,NaN,200.0,20.0,20.0,20.0
154,46b48281eb6d663ced748f324108c733,NaN,NaN,NaN,NaN,18500.0,41.0,30.0,41.0
197,5fb61f482620cb672f5e586bb132eae9,NaN,NaN,NaN,NaN,300.0,35.0,7.0,12.0
...,...,...,...,...,...,...,...,...,...
32515,b0a0c5dd78e644373b199380612c350a,NaN,NaN,NaN,NaN,1800.0,30.0,20.0,70.0
32589,10dbe0fbaa2c505123c17fdc34a63c56,NaN,NaN,NaN,NaN,800.0,30.0,10.0,23.0
32616,bd2ada37b58ae94cc838b9c0569fecd8,NaN,NaN,NaN,NaN,200.0,21.0,8.0,16.0
32772,fa51e914046aab32764c41356b9d4ea4,NaN,NaN,NaN,NaN,1300.0,45.0,16.0,45.0


In [16]:
products["product_category_name"] = products["product_category_name"].fillna("unknown")

In [17]:
products["product_name_lenght"] = products["product_name_lenght"].fillna(products["product_name_lenght"].median())

In [18]:
products["product_description_lenght"] = products["product_description_lenght"].fillna(products["product_description_lenght"].median())

In [19]:
products["product_photos_qty"] = products["product_photos_qty"].fillna(products["product_photos_qty"].median())

In [20]:
products["product_weight_g"] = products["product_weight_g"].fillna(products["product_weight_g"].median())

In [21]:
products["product_length_cm"] = products["product_length_cm"].fillna(products["product_length_cm"].median())

In [22]:
products["product_height_cm"] = products["product_height_cm"].fillna(products["product_height_cm"].median())

In [23]:
products["product_width_cm"] = products["product_width_cm"].fillna(products["product_width_cm"].median())

In [24]:
product_category_name[product_category_name.isna().any(axis=1)]

,product_category_name,product_category_name_english


In [25]:
category_name = pd.DataFrame({
    "product_category_name": ["pc_gamer", "portateis_cozinha_e_preparadores_de_alimentos", "unknown"],
    "product_category_name_english": ["pc_gamer", "portable_kitchen_and_food_preparers", "unknown"]
})

In [26]:
product_category_name = pd.concat([product_category_name, category_name], ignore_index=True)

In [27]:
product_category_name.info()

<class 'pandas.DataFrame'>
RangeIndex: 74 entries, 0 to 73
Data columns (total 2 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   product_category_name          74 non-null     str  
 1   product_category_name_english  74 non-null     str  
dtypes: str(2)
memory usage: 1.3 KB


In [28]:
products

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0
...,...,...,...,...,...,...,...,...,...
32946,a0b7d5a992ccda646f2d34e418fff5a0,moveis_decoracao,45.0,67.0,2.0,12300.0,40.0,40.0,40.0
32947,bf4538d88321d0fd4412a93c974510e6,construcao_ferramentas_iluminacao,41.0,971.0,1.0,1700.0,16.0,19.0,16.0
32948,9a7c6041fa9592d9d9ef6cfe62a71f8c,cama_mesa_banho,50.0,799.0,1.0,1400.0,27.0,7.0,27.0
32949,83808703fc0706a22e264b9d75f04a2e,informatica_acessorios,60.0,156.0,2.0,700.0,31.0,13.0,20.0


In [29]:
products = products.merge(product_category_name, on="product_category_name", how="left" )

In [30]:
products.isna().sum()

product_id                       0
product_category_name            0
product_name_lenght              0
product_description_lenght       0
product_photos_qty               0
product_weight_g                 0
product_length_cm                0
product_height_cm                0
product_width_cm                 0
product_category_name_english    0
dtype: int64

## 2.4 Deduplicating Geolocation

In [31]:
geolocation = geolocation.groupby("geolocation_zip_code_prefix", as_index=False).agg(
    {
        "geolocation_lat": "mean",
        "geolocation_lng": "mean",
        "geolocation_city": "first",
        "geolocation_state": "first"
    }
)

In [32]:
geolocation.shape

(19015, 5)

In [33]:
geolocation.nunique()

geolocation_zip_code_prefix    19015
geolocation_lat                19002
geolocation_lng                19001
geolocation_city                5955
geolocation_state                 27
dtype: int64

## Aggregating Order Items & payments

In [34]:
order_items_agg = order_items.groupby("order_id", as_index=False).agg(
    total_items=("order_item_id", "count"),
    total_price=("price", "sum"),
    total_freight=("freight_value", "sum")
)

In [35]:
order_items_agg.nunique()

order_id         98666
total_items         17
total_price       7761
total_freight     7970
dtype: int64

In [36]:
order_payments_agg = order_payments.groupby("order_id", as_index=False).agg(
    total_payment_value=("payment_value", "sum"),
    payment_installments=("payment_installments", "max"),
    payment_type=("payment_type", "first")
)

In [37]:
order_payments_agg.nunique()

order_id                99440
total_payment_value     27979
payment_installments       24
payment_type                5
dtype: int64

## 2. 6 Building & Saving Master Analytical Tables

In [38]:
order_reviews_agg = order_reviews.groupby("order_id", as_index=False).agg({
    "review_score": "mean",
    "review_comment_message": "first"
})

In [39]:
order_reviews_agg.nunique()

order_id                  98673
review_score                 11
review_comment_message    36106
dtype: int64

In [40]:
order_masters = orders.merge(customers, on="customer_id", how="left").merge(
    order_items_agg, on="order_id", how="left"
).merge(order_payments_agg, on="order_id", how="left").merge(
    order_reviews_agg, on="order_id", how="left"
)

In [41]:
order_masters.shape

(99441, 20)

In [42]:
item_masters = order_items.merge(products, on="product_id", how="left").merge(
    sellers, on="seller_id", how="left").merge(orders, on="order_id", how="left")

In [43]:
item_masters.shape

(112650, 26)

In [44]:
order_masters.to_csv(PROCESSED_DIR / "order_masters.csv", index=False)

In [45]:
item_masters.to_csv(PROCESSED_DIR / "item_masters.csv", index=False)